# 8장 실습 — 수요 만들기

지금까지는 저장소에 들어 있던 수요를 그대로 썼습니다. 이번에는 직접 만듭니다.
기말 프로젝트에서 여러분의 동네 수요를 만들 때 쓰는 절차입니다. 교재 8장에 대응합니다.

이 노트북에서 하는 일은 넷입니다.

1. 실제 O-D 데이터에서 시간대·공간 패턴을 봅니다 (교재 8.1 ~ 8.3)
2. 시뮬레이터가 받는 수요 형식을 확인합니다 (교재 8.4)
3. 수요를 세 단계로 만듭니다. 경계 안 균등 → 도로 위 → 시간대 프로파일 (교재 8.5 ~ 8.7)
4. 만든 수요를 11장의 시뮬레이션 루프에 넣어 결과가 어떻게 달라지는지 봅니다

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. O-D 데이터 (교재 8.1 ~ 8.3)

수도권 생활이동 데이터는 통신사 기지국 기록으로 만든 O-D 표입니다. 행 하나가 "이 동에서 저 동으로, 이 시간대에, 몇 명"입니다.
개인 기록이 아니라 집계값입니다. 하남시가 걸린 행만 뽑아 두었습니다.

In [ ]:
import pandas as pd
from smartmob.data import data_path

od = pd.read_parquet(data_path("hanam/od_2024.parquet"))
print(f"{len(od):,}행")
od.head(3)

`O_ADMDONG_CD` 와 `D_ADMDONG_CD` 가 출발·도착 행정동 코드입니다.
`ST_TIME_CD` 가 출발 시간대(0~23시), `CNT` 가 통행량(명)입니다.
먼저 시간대별 통행량을 봅니다. `groupby("ST_TIME_CD")["CNT"].sum()` 이 시간대마다 `CNT` 를 더합니다.

In [ ]:
import matplotlib.pyplot as plt

by_hour = od.groupby("ST_TIME_CD")["CNT"].sum()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(by_hour.index, by_hour.values, color="#4C6EF5", width=0.7)
ax.set_xlabel("출발 시각 (시)")
ax.set_ylabel("통행량 (명)")
ax.set_xticks(range(0, 24, 2))
ax.set_title("하남 O-D 통행량의 시간대 분포")
plt.tight_layout();

봉우리가 둘입니다. 오전 8시가 가장 높고 오후 5~6시가 두 번째입니다. 출퇴근입니다. 새벽 3~4시가 가장 낮습니다.

하남에서 출발한 통행이 어디로 가는지도 봅니다. 행정동 코드는 여덟 자리이고 앞 다섯 자리가 시군구입니다.
`// 1000` 은 정수 나눗셈이라 뒤 세 자리(동)를 떼고 시군구 코드만 남깁니다. 하남시는 41450 입니다.

In [ ]:
from_hanam = od[od["O_ADMDONG_CD"] // 1000 == 41450]                 # 하남에서 출발한 행
inside = from_hanam[from_hanam["D_ADMDONG_CD"] // 1000 == 41450]    # 그중 하남 안에서 끝난 행

total, stay = from_hanam["CNT"].sum(), inside["CNT"].sum()
print(f"하남 출발 총 통행  {total:>10,.0f}명")
print(f"하남 안에서 끝남   {stay:>10,.0f}명 ({stay / total:.0%})")
print(f"하남 행정동 수     {from_hanam['O_ADMDONG_CD'].nunique():>10}개")

하루 571,369명 중 308,002명, 54%가 하남 안에서 끝납니다. 나머지 절반은 서울 동남권과 인접 시군으로 나갑니다.
하남시 경계 안만 다루면 통행의 절반을 놓칩니다. 어디까지 자를지는 답하려는 질문에 따라 정합니다.

## 2. 시뮬레이터가 받는 형식 (교재 8.4)

시뮬레이터는 집계표가 아니라 개인 목록을 받습니다. 컬럼 다섯 개가 계약입니다.
`request_time` 은 자정부터의 분입니다. 1080 이면 18:00 입니다.

In [ ]:
from smartmob.data import DEMAND_COLUMNS, load_demand, validate_demand

print("필수 컬럼:", DEMAND_COLUMNS)

demand = load_demand("hanam")
print(demand.shape)
demand.head()

저장소의 하남 수요는 1,000건이고 `request_time` 이 1081~1439, 즉 저녁 6시부터 자정까지입니다.
`validate_demand` 가 형식을 검사합니다. 분을 초로 잘못 넣거나 좌표를 (경도, 위도) 순으로 넣는 실수를 여기서 잡습니다.

In [ ]:
validate_demand(demand)
print("[v] 형식 통과")

## 3. 1단계 — 경계 안에 균등하게 (교재 8.5)

가장 단순한 방법입니다. 시군구 경계 안에 점을 고르게 뿌립니다.
`generate_demand` 는 `boundary` 를 주면 경계 안에서, `graph` 를 주면 도로 위에서 뽑습니다. `hourly=None` 은 시각을 균등하게 뽑는다는 뜻입니다.
시간 범위는 기본값이 저장소 수요와 같은 저녁 6시~자정(1080~1440분)입니다.

In [ ]:
from smartmob.data import load_sigungu
from smartmob.teaching.demand_gen import generate_demand

boundary = load_sigungu("하남시")
flat = generate_demand(boundary=boundary, n=1000, seed=42, hourly=None)

validate_demand(flat)
print(flat.shape)
flat.head(3)

만든 수요도 실제 수요와 같은 다섯 컬럼이고 같은 검사를 통과합니다.
문제가 있습니다. 하남시에는 검단산과 한강이 있는데, 산과 강 위에서도 호출이 생깁니다.
승객이 산 한가운데에서 택시를 부르면 시뮬레이터가 그 지점을 도로에 스냅하느라 엉뚱한 곳으로 보냅니다.

## 4. 2단계 — 도로 위에 (교재 8.6)

도로망의 엣지 위에서 점을 뽑으면 이 문제가 사라집니다. 엣지 길이에 비례해 뽑으므로 큰길 주변에 더 많이 생깁니다.
두 방법의 출발지를 나란히 찍어 비교합니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))
on_road = generate_demand(graph=G, n=1000, seed=42, hourly=None)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=True, sharey=True)
for ax, df, title in [(axes[0], flat, "경계 안 균등"), (axes[1], on_road, "도로 위")]:
    ax.scatter(df["origin_lon"], df["origin_lat"], s=4, alpha=0.4, color="#4C6EF5")
    ax.set_title(title)
    ax.set_xlabel("경도")
    ax.set_aspect(1 / 0.79)
axes[0].set_ylabel("위도")
plt.tight_layout();

오른쪽은 산과 강이 비고 도로가 촘촘한 시가지에 점이 몰립니다.
숨은 가정이 하나 있습니다. 도로가 많은 곳에 사람이 많다는 것입니다. 고속도로 구간처럼 도로는 길지만 택시를 부르지 않는 곳도 있습니다.

## 5. 3단계 — 시간대 프로파일 (교재 8.7)

지금까지 만든 수요는 시간에 대해 균등합니다. 1절에서 본 출퇴근 첨두가 없습니다.
`hourly_profile_from_od` 가 O-D 표에서 시간대 비중 24개를 뽑습니다. 합이 1이 되도록 정규화한 값입니다.

In [ ]:
from smartmob.teaching.demand_gen import HANAM_HOURLY, hourly_profile_from_od

profile = hourly_profile_from_od(od)
for hour in (3, 8, 12, 17, 22):
    print(f"{hour:2d}시  {profile[hour]:.1%}   (HANAM_HOURLY {HANAM_HOURLY[hour]:.1%})")

하루 통행의 7.9%가 8시대에 몰리고 3시대는 0.6%입니다. 열세 배 차이입니다.
`HANAM_HOURLY` 는 같은 계산을 미리 해서 패키지에 넣어 둔 값이라 소수 셋째 자리까지 같습니다. 이 비율대로 호출 시각을 뽑습니다.
저녁 6시~자정 범위에서 만들어, 균등하게 뽑은 것과 시간대별 건수를 비교합니다.

In [ ]:
realistic = generate_demand(graph=G, n=1000, seed=42, hourly=profile)

fig, ax = plt.subplots(figsize=(8, 3.5))
for df, label in [(on_road, "고르게"), (realistic, "실제 프로파일")]:
    counts = df["request_time"].floordiv(60).value_counts().sort_index()   # 분 → 시 로 묶어 세기
    ax.plot(counts.index, counts.values, marker="o", label=label)
ax.set_xlabel("시각 (시)")
ax.set_ylabel("호출 수")
ax.legend()
plt.tight_layout();

같은 1,000건인데 실제 프로파일 쪽은 18시에 몰리고 23시로 갈수록 줄어듭니다. 8.2절 그래프의 저녁 부분과 모양이 같습니다.
시간만 실제 데이터를 따랐습니다. 공간은 여전히 도로 길이에 비례해 뽑았고, 1절에서 본 O-D 쌍은 쓰지 않았습니다. 출발지와 목적지를 쌍으로 뽑는 것은 연습 8.2 입니다.

## 6. 만든 수요로 시뮬레이션 돌리기

수요를 바꾸면 결과가 어떻게 달라지는지 직접 봅니다. 11장의 루프 `simulate` 를 미리 빌려 씁니다.
인자는 수요, 차량, 시작 분, 끝 분입니다. 서버 없이 로컬에서 몇 초 안에 돕니다.

In [ ]:
from smartmob.data import load_vehicles
from smartmob.teaching.simloop import simulate

vehicles = load_vehicles("hanam")
print(f"차량 {len(vehicles)}대")

runs = {}
for label, df in [("고르게 흩뿌린 수요", on_road), ("실제 프로파일 수요", realistic)]:
    runs[label] = simulate(df, vehicles, 1080, 1440)

pd.DataFrame({
    label: {
        "서비스율": round(r.summary()["service_rate"], 3),
        "평균대기_분": round(r.summary()["avg_waiting_time_min"], 2),
        "최대대기_분": round(r.summary()["max_waiting_time_min"], 1),
        "가동률": round(r.summary()["utilization"], 3),
    }
    for label, r in runs.items()
}).T

같은 차량 80대, 같은 호출 1,000건인데 서비스율이 99.9%에서 92.2%로 떨어지고 평균 대기가 5.2분에서 6.8분으로 늘어납니다.
수요가 18시에 몰리면 그 시간대에 차가 모자라기 때문입니다. 총량이 아니라 분포가 서비스 수준을 정합니다.

## 7. 수요 조건 바꾸기

### 7.1 나만의 시간대 프로파일

24개짜리 리스트를 만들어 넣습니다. 합이 1이 아니어도 됩니다. 안에서 정규화합니다.
저녁 9시에 극단적으로 몰리는 프로파일을 만들어 서비스율이 얼마나 떨어지는지 봅니다.

In [ ]:
my_hourly = None      # 길이 24의 리스트. 예: [0]*18 + [1, 5, 2, 1] + [0, 0]

banner("빈칸 7.1")
if my_hourly:
    peaky = generate_demand(graph=G, n=1000, seed=42, hourly=tuple(my_hourly))
    run = simulate(peaky, vehicles, 1080, 1440)
    s = run.summary()
    print(f"[v] 서비스율 {s['service_rate']:.1%}, "
          f"평균대기 {s['avg_waiting_time_min']:.2f}분, "
          f"최대대기 {s['max_waiting_time_min']:.1f}분")
else:
    print("[ ] my_hourly 가 비어 있습니다")

### 7.2 서비스율이 90% 미만이 되는 요청 건수

호출 건수를 1,000건에서 늘려 가며 서비스율이 90% 아래로 떨어지는 지점을 찾습니다. 차량은 80대 그대로 둡니다.
아래 실험용 코드의 건수를 바꿔 가며 돌립니다. 1,000건에서 92.2%, 2,000건에서 62.0%이므로 그 사이 어딘가입니다.

In [ ]:
break_point = None      # 서비스율이 90% 아래로 내려가는 호출 건수

# 실험용 코드입니다. 건수를 바꿔 가며 돌립니다.
for n in [1000, 2000, 3000]:
    df = generate_demand(graph=G, n=n, seed=42, hourly=profile)
    s = simulate(df, vehicles, 1080, 1440).summary()
    print(f"호출 {n:5,}건 → 서비스율 {s['service_rate']:.1%}, "
          f"평균대기 {s['avg_waiting_time_min']:.2f}분")

banner("빈칸 7.2")
todo("서비스율이 90% 아래로 내려가는 호출 건수", break_point)

### 7.3 seed 를 바꾸면

`seed` 만 바꿔 다섯 번 돌려 서비스율의 표준편차를 구합니다.
루프 자체는 결정론적이고, 난수는 수요 생성에 있습니다. seed 를 고정하지 않으면 두 시나리오의 차이가 난수 차이인지 알 수 없습니다.

In [ ]:
service_rate_std = None     # seed 5개에서 나온 서비스율의 표준편차

banner("빈칸 7.3")
todo("서비스율 표준편차", service_rate_std, fmt=lambda v: f"{v:.4f}")

## 정리

- O-D 표는 "어디서 어디로 몇 명"을 담은 집계 자료입니다. 하남 통행의 54%가 하남 안에서 끝납니다
- 수요는 컬럼 다섯 개짜리 표입니다. `validate_demand` 로 형식을 먼저 확인합니다
- 경계 안 균등 → 도로 위 → 시간대 프로파일 순으로 현실에 가까워집니다. 8시대 7.9%, 3시대 0.6%입니다
- 총 호출 건수가 같아도 시간대 분포가 다르면 서비스율이 99.9%에서 92.2%로 달라집니다
- 난수는 수요 생성에 있습니다. seed 를 고정하지 않으면 비교가 무의미해집니다
- 9장 실습에서는 배차에 필요한 도착 예상시간을 모델로 예측합니다